# OSB-JailbreakBench — Main Analysis

Figures, tables, and taxonomy for the benchmark report.

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='white')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FIGURES_DIR = '../results/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

## Load Scored Results

In [ ]:
scores_dir = '../results/scores'
all_scores = []

for filename in sorted(os.listdir(scores_dir)):
    if filename.endswith('_scored.json'):
        with open(os.path.join(scores_dir, filename), encoding='utf-8') as f:
            all_scores.extend(json.load(f))

df = pd.DataFrame(all_scores)
df_valid = df[df['binary_score'].notna()].copy()
print(f'Total records: {len(df)}')
print(f'Valid (scoreable) records: {len(df_valid)}')
print(f'Models: {sorted(df_valid["model"].unique())}')
print(f'Categories: {sorted(df_valid["category"].unique())}')

## ASR Table

In [ ]:
asr = (
    df_valid
    .groupby(['model', 'category'])['binary_score']
    .mean()
    .unstack()
    .round(3)
)

print('\n=== Full ASR Table ===')
print(asr.to_string())

## Figure 1 — ASR Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

sns.heatmap(
    asr,
    annot=True,
    fmt='.2f',
    cmap='Reds',
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Attack Success Rate', 'shrink': 0.8},
    ax=ax,
)

ax.set_title('Attack Success Rate by Model and Category', fontsize=14, fontweight='bold', pad=16)
ax.set_xlabel('Attack Category', fontsize=11)
ax.set_ylabel('Model', fontsize=11)
ax.tick_params(axis='x', rotation=20)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
out1 = os.path.join(FIGURES_DIR, 'asr_heatmap.png')
plt.savefig(out1, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out1}')

## Figure 2 — Mean ASR by Model

In [ ]:
mean_by_model = asr.mean(axis=1).sort_values(ascending=True)  # ascending=True so most robust is at top

print('\n=== Mean ASR per Model (most robust first) ===')
for model, val in mean_by_model.items():
    print(f'  {model:<22} {val:.3f}')

fig, ax = plt.subplots(figsize=(8, 4))

colors = ['#d73027' if v == mean_by_model.max() else '#4393c3' for v in mean_by_model.values]
bars = ax.barh(mean_by_model.index, mean_by_model.values, color=colors, edgecolor='white', height=0.55)

for bar, val in zip(bars, mean_by_model.values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlim(0, max(mean_by_model.values) * 1.25)
ax.set_xlabel('Mean Attack Success Rate', fontsize=11)
ax.set_title('Mean ASR by Model (all categories)', fontsize=13, fontweight='bold', pad=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=10)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

plt.tight_layout()
out2 = os.path.join(FIGURES_DIR, 'mean_asr_by_model.png')
plt.savefig(out2, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out2}')

## Figure 3 — Mean ASR by Category

In [ ]:
mean_by_cat = asr.mean(axis=0).sort_values(ascending=True)  # ascending=True so highest is at top

print('\n=== Mean ASR per Category (highest first) ===')
for cat, val in mean_by_cat.sort_values(ascending=False).items():
    print(f'  {cat:<25} {val:.3f}')

fig, ax = plt.subplots(figsize=(8, 4))

colors = ['#d73027' if v == mean_by_cat.max() else '#4393c3' for v in mean_by_cat.values]
bars = ax.barh(mean_by_cat.index, mean_by_cat.values, color=colors, edgecolor='white', height=0.55)

for bar, val in zip(bars, mean_by_cat.values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlim(0, max(mean_by_cat.values) * 1.3)
ax.set_xlabel('Mean Attack Success Rate', fontsize=11)
ax.set_title('Mean ASR by Attack Category (all models)', fontsize=13, fontweight='bold', pad=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=10)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

plt.tight_layout()
out3 = os.path.join(FIGURES_DIR, 'mean_asr_by_category.png')
plt.savefig(out3, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out3}')

## Summary Statistics

In [ ]:
# Highest single cell
flat = asr.stack()
max_idx = flat.idxmax()
max_val = flat.max()

# Lowest non-zero cell
nonzero = flat[flat > 0]
min_idx = nonzero.idxmin()
min_val = nonzero.min()

print('\n=== Summary Statistics ===')
print(f'Highest single ASR cell: {max_idx[0]} / {max_idx[1]} = {max_val:.3f}')
print(f'Lowest non-zero ASR cell: {min_idx[0]} / {min_idx[1]} = {min_val:.3f}')
print(f'\nFigures saved:')
for path in [out1, out2, out3]:
    size_kb = os.path.getsize(path) // 1024
    print(f'  {path}  ({size_kb} KB)')